# Transformer Inference Walkthrough
This notebook provides a block-by-block visual breakdown of the `export_decisions.py` pipeline. It imports a single vehicle's CSV, normalizes the data, runs the Transformer forward pass, and manually evaluates Loop A and Loop B.

In [1]:
import os
import torch
import numpy as np
import pandas as pd
import json
import glob

device = torch.device('cpu')
experiment_folder = 'experiments/finetune_intent_2026-05-20_13-48-21_WP9'
data_dir = 'resources/poc_dataset/raw'
csv_files = sorted(glob.glob(f'{data_dir}/*.csv'))
data_file = csv_files[0] if csv_files else None

print(f'Experiment: {experiment_folder}')
print(f'Target CSV: {data_file}')

Experiment: experiments/finetune_intent_2026-05-20_13-48-21_WP9
Target CSV: resources/poc_dataset/raw/data_car_294_t18183.csv


### 1. Load Configuration and Transformer Architecture

In [2]:
with open(os.path.join(experiment_folder, 'configuration.json')) as f:
    config = json.load(f)

from src.transformer_model.encoder import TSTransformerEncoder

model = TSTransformerEncoder(
    feat_dim=51,
    max_len=config.get('max_seq_len', 60),
    embedding_dim=config['embedding_dim'],
    n_heads=config['num_heads'],
    num_layers=config['num_layers'],
    hidden_dim=config['hidden_dim'],
    dropout=config['dropout'],
    pos_encoding=config['pos_encoding'],
    activation=config['activation'],
    norm=config['normalization_layer'],
    num_intents=config.get('num_intents', 4),
)

ckpt_path = os.path.join(experiment_folder, 'checkpoints', 'model_best.pth')
state = torch.load(ckpt_path, map_location=device)
state_dict = state.get('state_dict', state)
clean_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(clean_state_dict, strict=False)
model.eval()
print('Transformer Model Loaded Successfully!')

Transformer Model Loaded Successfully!


/home/massi/myStuff/dev/VANET_Trajectory_Pred_With_Transformers/src/transformer_model/encoder.py:279: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer was not TransformerEncoderLayer
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)


### 2. Pre-Compute Global Physics Bounds (Normalization Engine)

In [3]:
global_min, global_max = None, None

for filepath in csv_files:
    df = pd.read_csv(filepath)
    if 'LaneID' in df.columns:
        df['LaneID'] = pd.factorize(df['LaneID'])[0]
    num_df = df.select_dtypes(include=[np.number])
    meta_cols = {'track_id', 'Time', 'file_id', 'VehicleID', 'car_id'}
    feature_cols = [c for c in num_df.columns if c not in meta_cols]
    feats = num_df[feature_cols].values.astype(np.float32)
    if feats.shape[1] == 51:
        f_min, f_max = feats.min(axis=0), feats.max(axis=0)
        global_min = f_min if global_min is None else np.minimum(global_min, f_min)
        global_max = f_max if global_max is None else np.maximum(global_max, f_max)

print(f'Global Bounds Calculated across {len(csv_files)} vehicles.')

Global Bounds Calculated across 5 vehicles.


### 3. Load and Normalize Target Vehicle Matrix

In [4]:
df = pd.read_csv(data_file)
times = df['Time'].values

if 'LaneID' in df.columns:
    df['LaneID'] = pd.factorize(df['LaneID'])[0]
num_df = df.select_dtypes(include=[np.number])
features = num_df[feature_cols].values.astype(np.float32)

# Normalize to [0.0, 1.0] matching Transformer Phase 1 training
features_norm = (features - global_min) / (global_max - global_min + 1e-8)

print(f'Feature matrix normalized: {features_norm.shape} (TimeSteps, Features)')

Feature matrix normalized: (10001, 51) (TimeSteps, Features)


### 4. Extract a Single 60-Step Window

In [5]:
seq_len = config.get('data_chunk_len', 60)
start_idx = 0

window = features_norm[start_idx : start_idx + seq_len]
window_tensor = torch.tensor(window).unsqueeze(0).to(device)  # Add Batch Dimension
padding_mask = torch.zeros((1, seq_len), dtype=torch.bool).to(device)

print(f'Input Tensor Shape: {window_tensor.shape}')

Input Tensor Shape: torch.Size([1, 60, 51])


### 5. Transformer Forward Pass (Inference)

In [6]:
with torch.no_grad():
    recon_output, intent_logits, _, _ = model(window_tensor, padding_mask)

prediction = recon_output[0].cpu().numpy()
intent_out = intent_logits[0].cpu().numpy()

print(f'Predicted Trajectory Output : {prediction.shape}')
print(f'Raw Intent Logits Output    : {intent_out}')

Predicted Trajectory Output : (60, 51)
Raw Intent Logits Output    : [-0.03341105 -0.05082342  0.07192676  0.06229904]


### 6. Loop B Engine (Intent Score & MAC Delay)

In [9]:
from src.loops.loop_b import StabilityScorer, MACBiasMapper
from src.deploy.attach_labels import VEHICLE_LABELS

scorer = StabilityScorer(constant=0.5)
mapper = MACBiasMapper(min_wait_ms=1.0, max_wait_ms=100.0)

loop_b = scorer.score(torch.tensor(intent_out).unsqueeze(0))
mac_wait_ms = mapper.map(loop_b.P_stable)

intent_idx = int(np.argmax(intent_out)) if np.any(intent_out) else 0
intent_label = VEHICLE_LABELS.get(intent_idx, 'MaintainLane')

print('--- Loop B Results ---')
print(f'Predicted Intent: {intent_label}')
print(f'P_stable Score  : {loop_b.P_stable:.4f}')
print(f'MAC Delay Bias  : {mac_wait_ms:.2f} ms')

--- Loop B Results ---
Predicted Intent: Exit
P_stable Score  : 0.2384
MAC Delay Bias  : 16.41 ms


### 7. Loop A Engine (Spatial Error Beacon Suppression)

In [10]:
from src.loops.loop_a import DiscrepancyMonitor

monitor = DiscrepancyMonitor(epsilon=0.5, beacon_hz_low=2.0, beacon_hz_high=10.0)

# Compare the prediction against the ACTUAL next 60 timesteps (t+1 ... t+60)
next_actual_window = features_norm[start_idx + seq_len : start_idx + seq_len + seq_len]

if len(next_actual_window) == seq_len:
    loop_a = monitor.check(predicted_window=prediction, actual_window=next_actual_window)
    print('--- Loop A Results ---')
    print(f'Mean L2 Error (Spatial): {loop_a.residual:.4f}')
    if loop_a.flag == 0:
        print(f'Status: STABLE (Prediction highly accurate) -> Beacon Suppressed to {loop_a.beacon_hz} Hz')
    else:
        print(f'Status: ALERT (Prediction drifted) -> Beacon Boosted to {loop_a.beacon_hz} Hz')
else:
    print('Not enough future timesteps remaining in the CSV to calculate Loop A residual.')

--- Loop A Results ---
Mean L2 Error (Spatial): 1.5684
Status: ALERT (Prediction drifted) -> Beacon Boosted to 10.0 Hz
